# K-Nearest Neighbors (KNN)

KNN is a non-parametric, instance-based algorithm. No training phase - classification happens at prediction time by finding the K closest training points.

1. **How KNN Works** - Distance metrics, K selection
2. **Effect of K** - Bias-variance tradeoff
3. **Distance Metrics** - Euclidean, Manhattan, Minkowski

**Dataset**: Iris (multi-class classification)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

sns.set_theme(style="whitegrid")

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Finding the optimal K
k_range = range(1, 31)
cv_scores = []

for k in k_range:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k)),
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    cv_scores.append(scores.mean())

best_k = k_range[np.argmax(cv_scores)]
print(f"Best K: {best_k} (CV accuracy: {max(cv_scores):.3f})")

plt.figure(figsize=(8, 5))
plt.plot(k_range, cv_scores, "o-", color="teal")
plt.axvline(x=best_k, color="coral", linestyle="--", label=f"Best K={best_k}")
plt.xlabel("K (number of neighbors)")
plt.ylabel("Cross-validated Accuracy")
plt.title("KNN: Accuracy vs K")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Final model with best K
pipe_best = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k)),
])
pipe_best.fit(X_train, y_train)

print(f"Test accuracy: {pipe_best.score(X_test, y_test):.3f}\n")
print(classification_report(y_test, pipe_best.predict(X_test), target_names=iris.target_names))

In [ ]:
# Decision boundary visualization (2D projection)
X_2d = X_train[:, :2]  # Use first 2 features for visualization

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [1, 5, 25]):
    knn = KNeighborsClassifier(n_neighbors=k)
    scaler = StandardScaler()
    X_2d_s = scaler.fit_transform(X_2d)
    knn.fit(X_2d_s, y_train)
    
    xx, yy = np.meshgrid(
        np.linspace(X_2d_s[:, 0].min() - 1, X_2d_s[:, 0].max() + 1, 200),
        np.linspace(X_2d_s[:, 1].min() - 1, X_2d_s[:, 1].max() + 1, 200),
    )
    Z = knn.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")
    ax.scatter(X_2d_s[:, 0], X_2d_s[:, 1], c=y_train, cmap="viridis", edgecolors="k", s=20)
    ax.set_title(f"K={k}")

plt.suptitle("KNN Decision Boundaries (K=1: overfit, K=25: smooth)")
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Scaling is critical** - KNN uses distance, so features on different scales will dominate
2. **Small K = low bias, high variance** (overfitting); **Large K = high bias, low variance** (underfitting)
3. **No training phase** - but prediction is O(n) per query (slow for large datasets)
4. **Curse of dimensionality** - KNN struggles in high dimensions; use PCA or feature selection first
5. **Good baseline** - simple, interpretable, no assumptions about data distribution